# Titanic survival — exploratory analysis and pipeline baselines

Execution checked on 2026-10-16 with the versions in `requirements.txt`; source outputs are cleared for a compact repository. The results are exploratory, not an independent final evaluation. See [Project Documentation](Project%20Documentation/README.md) for the run summary and limitations.

This notebook removes two rows with missing `Embarked` before a stratified 80/20 split (`random_state=42`): 711 training and 178 evaluation rows. Imputation and encoding are inside model pipelines; ticket frequencies are EDA only. Full-dataset EDA precedes splitting, and multiple models use the same evaluation subset. The other notebook uses a different preparation order and split.

**Code assistance:** The code was developed with assistance from **CODEX**.


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("Data/Titanic_dataset.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df[df["Fare"]==0]

In [ ]:
df[df["Fare"] == 0].shape

In [ ]:
df["Survived"].value_counts()

In [ ]:
df["Survived"].value_counts(normalize=True) * 100

In [ ]:
df["Sex"].value_counts()

In [ ]:
df.groupby("Sex")["Survived"].mean() * 100

In [ ]:
df.groupby("Pclass")["Survived"].mean() * 100

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df["Age"].hist(bins=20)

plt.xlabel("Age")
plt.ylabel("Number of passengers")
plt.title("Age Distribution")
plt.show()

In [ ]:
df[df["Survived"] == 0]["Age"].hist(
    bins=20,
    alpha=0.5,
    label="Did not survive"
)

df[df["Survived"] == 1]["Age"].hist(
    bins=20,
    alpha=0.5,
    label="Survived"
)

plt.xlabel("Age")
plt.ylabel("Number of passengers")
plt.title("Age Distribution by Survival")
plt.legend()
plt.show()

In [ ]:
df[df["Embarked"].isnull()][["PassengerId", "Pclass", "Sex", "Age", "Fare", "Embarked"]]

In [ ]:
df[df["Fare"] == 80][["PassengerId", "Pclass", "Sex", "Age", "Fare", "Embarked"]]

In [ ]:
df[df["Fare"] == 80]

In [ ]:
df[
    (df["Pclass"] == 1) &
    (df["Fare"] >= 70) &
    (df["Fare"] <= 90) &
    (df["Embarked"].notnull())
][["Pclass", "Fare", "Cabin", "Ticket", "Embarked"]]

In [ ]:
df[
    (df["Pclass"] == 1) &
    (df["Fare"] >= 70) &
    (df["Fare"] <= 90) &
    (df["Embarked"].notnull())
]["Embarked"].value_counts()

In [ ]:
df["PassengerId"].head(10)

In [ ]:
df["Name"].head(10)

In [ ]:
df["Name"].str.extract(r",\s*([^.]*)\.")[0].value_counts()

In [ ]:
df["SibSp"].value_counts().sort_index()

In [ ]:
df["Parch"].value_counts().sort_index()

In [ ]:
family_size = df["SibSp"] + df["Parch"] + 1

In [ ]:
family_size.value_counts().sort_index()

In [ ]:
df.groupby(family_size)["Survived"].mean() * 100

In [ ]:
df["Ticket"].nunique()

In [ ]:
df["Ticket"].value_counts().head(10)

In [ ]:
ticket_group_size = df["Ticket"].map(df["Ticket"].value_counts())
ticket_group_size.value_counts().sort_index()

In [ ]:
df.groupby(ticket_group_size)["Survived"].mean() * 100

**Baseline note:** `TicketGroupSize` is excluded from modeling because its original implementation used full-dataset ticket frequencies before the split. The ticket-frequency analysis above is EDA only.


In [ ]:
df.groupby("Embarked")["Survived"].mean() * 100

In [ ]:
## Feature Engineering
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

In [ ]:
df[["SibSp", "Parch", "FamilySize"]].head(10)

In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")

In [ ]:
df["Title"] = df["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

In [ ]:
main_titles = ["Mr", "Mrs", "Miss", "Master"]

df.loc[~df["Title"].isin(main_titles), "Title"] = "Rare"

In [ ]:
df = df.dropna(subset=["Embarked"])

In [ ]:
df["Embarked"].isnull().sum()

In [ ]:
df = df.drop(columns=[
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin"
])

In [ ]:
df.columns

In [ ]:
X = df.drop(columns=["Survived"])
y = df["Survived"]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title"
]

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

In [ ]:
numeric_features = [
    "Pclass",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "FamilySize"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title"
]

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [ ]:
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Did not survive", "Survived"]
)

disp.plot()
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Did not survive", "Survived"]
    )
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Predictions
y_pred = model.predict(X_test)

# Predicted probabilities for the positive class (Survived = 1)
y_prob = model.predict_proba(X_test)[:, 1]

# Main metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

# Display main metrics
print("MODEL EVALUATION")
print("------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

# Classification report
print("\nCLASSIFICATION REPORT")
print("------------------------")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Did not survive", "Survived"]
    )
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Did not survive", "Survived"]
)

disp.plot()
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Predictions
y_pred = model.predict(X_test)

# Predicted probabilities for the positive class (Survived = 1)
y_prob = model.predict_proba(X_test)[:, 1]

# Main metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

# Display main metrics
print("MODEL EVALUATION")
print("------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

# Classification report
print("\nCLASSIFICATION REPORT")
print("------------------------")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Did not survive", "Survived"]
    )
)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Did not survive", "Survived"]
)

disp.plot()
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42
    ))
])

random_forest_model.fit(X_train, y_train)

In [ ]:
rf_y_pred = random_forest_model.predict(X_test)
rf_y_prob = random_forest_model.predict_proba(X_test)[:, 1]

In [ ]:
rf_accuracy = accuracy_score(y_test, rf_y_pred)
rf_precision = precision_score(y_test, rf_y_pred)
rf_recall = recall_score(y_test, rf_y_pred)
rf_f1 = f1_score(y_test, rf_y_pred)
rf_roc_auc = roc_auc_score(y_test, rf_y_prob)

print("RANDOM FOREST - MODEL EVALUATION")
print("--------------------------------")
print(f"Accuracy:  {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall:    {rf_recall:.4f}")
print(f"F1-score:  {rf_f1:.4f}")
print(f"ROC-AUC:   {rf_roc_auc:.4f}")

print("\nCLASSIFICATION REPORT")
print("--------------------------------")
print(
    classification_report(
        y_test,
        rf_y_pred,
        target_names=["Did not survive", "Survived"]
    )
)

rf_cm = confusion_matrix(y_test, rf_y_pred)

rf_disp = ConfusionMatrixDisplay(
    confusion_matrix=rf_cm,
    display_labels=["Did not survive", "Survived"]
)

rf_disp.plot()
plt.title("Confusion Matrix - Random Forest")
plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

In [ ]:
cv_results = grid_search.cv_results_

for c, score in zip(
    cv_results["param_classifier__C"],
    cv_results["mean_test_score"]
):
    print(f"C = {c}: ROC-AUC = {score:.4f}")

In [ ]:
from sklearn.model_selection import GridSearchCV

rf_param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 5, 10],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

rf_grid_search = GridSearchCV(
    estimator=random_forest_model,
    param_grid=rf_param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

rf_grid_search.fit(X_train, y_train)

In [ ]:
print("Best parameters:")
print(rf_grid_search.best_params_)

print("\nBest CV ROC-AUC:")
print(rf_grid_search.best_score_)

In [ ]:
# Best Random Forest selected by GridSearchCV
best_rf_model = rf_grid_search.best_estimator_

# Predictions on test
best_rf_y_pred = best_rf_model.predict(X_test)
best_rf_y_prob = best_rf_model.predict_proba(X_test)[:, 1]

# Metrics
best_rf_accuracy = accuracy_score(y_test, best_rf_y_pred)
best_rf_precision = precision_score(y_test, best_rf_y_pred)
best_rf_recall = recall_score(y_test, best_rf_y_pred)
best_rf_f1 = f1_score(y_test, best_rf_y_pred)
best_rf_roc_auc = roc_auc_score(y_test, best_rf_y_prob)

print("OPTIMIZED RANDOM FOREST - TEST EVALUATION")
print("-----------------------------------------")
print(f"Accuracy:  {best_rf_accuracy:.4f}")
print(f"Precision: {best_rf_precision:.4f}")
print(f"Recall:    {best_rf_recall:.4f}")
print(f"F1-score:  {best_rf_f1:.4f}")
print(f"ROC-AUC:   {best_rf_roc_auc:.4f}")

print("\nCLASSIFICATION REPORT")
print("-----------------------------------------")
print(
    classification_report(
        y_test,
        best_rf_y_pred,
        target_names=["Did not survive", "Survived"]
    )
)

# Confusion Matrix
best_rf_cm = confusion_matrix(y_test, best_rf_y_pred)

best_rf_disp = ConfusionMatrixDisplay(
    confusion_matrix=best_rf_cm,
    display_labels=["Did not survive", "Survived"]
)

best_rf_disp.plot()
plt.title("Confusion Matrix - Optimized Random Forest")
plt.show()